<a href="https://www.kaggle.com/code/muneeburrehman98/deimv2-finetune-anime?scriptVersionId=333378518" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!git clone https://github.com/Intellindust-AI-Lab/DEIMv2.git

Cloning into 'DEIMv2'...
remote: Enumerating objects: 581, done.
remote: Counting objects: 100% (327/327), done.
remote: Compressing objects: 100% (229/229), done.
remote: Total 581 (delta 156), reused 98 (delta 98), pack-reused 254 (from 1)
Receiving objects: 100% (581/581), 1.70 MiB | 8.44 MiB/s, done.
Resolving deltas: 100% (191/191), done.


In [2]:
!pip install -q faster-coco-eval gdown onnx onnxsim onnxscript calflops tensorboard
!pip install -q -r DEIMv2/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [3]:
!hf download Intellindust/DEIMv2_DINOv3_L_COCO --local-dir ./ckpts

A new version of huggingface_hub (1.22.0) is available! You are using version 1.4.1.
To update, run: pip install -U huggingface_hub

Fetching 4 files: 100%|███████████████████████████| 4/4 [00:06<00:00,  1.71s/it]
Download complete: : 131MB [00:06, 18.8MB/s]                                    /kaggle/working/ckpts
Download complete: : 131MB [00:06, 19.0MB/s]


In [4]:
import torch
from safetensors.torch import load_file

weights = load_file("ckpts/model.safetensors")

checkpoint = {
    "model": weights
}

torch.save(checkpoint, "deimv2.pth")

print("Successfully converted model.safetensors to deimv2.pth")

Successfully converted model.safetensors to deimv2.pth


In [5]:
!kaggle datasets download muneeburrehman98/danbooru-annotated-images

Dataset URL: https://www.kaggle.com/datasets/muneeburrehman98/danbooru-annotated-images
License(s): MIT
100%|███████████████████████████████████████| 1.64G/1.64G [00:13<00:00, 132MB/s]



In [6]:
!unzip -q danbooru-annotated-images.zip

In [7]:
import json
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

root_dir = Path("dataset")
source_images_dir = root_dir / "images"  
target_dir = root_dir 
seed = 42

master_json = root_dir / "annotations.json"

anno_dir = target_dir / "annotations"
images_output_dir = target_dir / "images"

split_names = {
    "train": "instances_train.json",
    "val": "instances_val.json",
    "test": "instances_test.json"
}

for split in ["train", "val", "test"]:
    (images_output_dir / split).mkdir(parents=True, exist_ok=True)
anno_dir.mkdir(parents=True, exist_ok=True)

if all((anno_dir / name).exists() for name in split_names.values()):
    print("Splits and directory structure already exist.")
    exit()

with open(master_json, 'r') as f:
    data = json.load(f)

images = data['images']
annotations = data['annotations']
categories = data['categories']

buckets = [img.get('bucket', 'unknown') for img in images]

train_imgs, temp_imgs, train_buckets, temp_buckets = train_test_split(
    images, buckets, test_size=0.30, stratify=buckets, random_state=seed
)

val_imgs, test_imgs = train_test_split(
    temp_imgs, test_size=0.50, stratify=temp_buckets, random_state=seed
)

def process_split(split_imgs, split_type):
    split_ids = {img['id'] for img in split_imgs}
    
    split_annos = [ann for ann in annotations if ann['image_id'] in split_ids]
    
    output_data = {
        "images": split_imgs,
        "annotations": split_annos,
        "categories": categories
    }
    json_path = anno_dir / split_names[split_type]
    with open(json_path, 'w') as f:
        json.dump(output_data, f, indent=4)

    print(f"Moving {split_type} images...")
    for img in split_imgs:
        file_name = img['file_name']
        src = source_images_dir / file_name
        dst = images_output_dir / split_type / file_name
        
        if src.exists():
            shutil.move(src, dst)
        else:
            print(f"Warning: File {src} not found.")

process_split(train_imgs, "train")
process_split(val_imgs, "val")
process_split(test_imgs, "test")

print("--- Organization Complete ---")
print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}")

Moving train images...
Moving val images...
Moving test images...
--- Organization Complete ---
Train: 10500 | Val: 2250 | Test: 2250


In [8]:
%%writefile deimv2_finetune.yml

__include__: [
  'deimv2_dinov3_l_coco.yml',
]

output_dir: ./outputs

num_classes: 1 
remap_mscoco_category: False

epoches: 32          # 20 for training + EMA for 4n = 12
flat_epoch: 14       # 4 + 20 // 2
no_aug_epoch: 12     # 4n

train_dataloader:
  total_batch_size: 24
  num_workers: 2   
  dataset:
    img_folder: dataset/images/train
    ann_file: dataset/annotations/instances_train.json
    transforms:
      ops:
        - {type: Mosaic, output_size: 320, rotation_range: 0, translation_range: [0.05, 0.05], scaling_range: [0.8, 1.2], probability: 1.0, fill_value: 0, use_cache: True, max_cached_images: 50, random_pop: True}
        - {type: RandomPhotometricDistort, p: 0.2} 
        - {type: RandomZoomOut, fill: 0}
        - {type: RandomIoUCrop, p: 0.5}            
        - {type: SanitizeBoundingBoxes, min_size: 1}
        - {type: RandomHorizontalFlip}             
        - {type: Resize, size: [640, 640]}
        - {type: SanitizeBoundingBoxes, min_size: 1}
        - {type: ConvertPILImage, dtype: 'float32', scale: True}
        - {type: Normalize, mean: [0.485, 0.456, 0.406], std: [0.229, 0.224, 0.225]}
        - {type: ConvertBoxes, fmt: 'cxcywh', normalize: True}
      policy:
        epoch: [4, 14, 20] 

  collate_fn:
    mixup_epochs: [4, 14]       
    stop_epoch: 20              
    copyblend_epochs: [4, 20]   

val_dataloader:
  total_batch_size: 24
  num_workers: 2
  dataset:
    img_folder: dataset/images/val
    ann_file: dataset/annotations/instances_val.json

DEIMCriterion:
  matcher:
    matcher_change_epoch: 18    

Writing deimv2_finetune.yml


In [9]:
%cp deimv2_finetune.yml DEIMv2/configs/deimv2/deimv2_finetune.yml

In [10]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True NCCL_P2P_DISABLE=1 \
    NCCL_IB_DISABLE=1 CUDA_VISIBLE_DEVICES=0,1 torchrun --nproc_per_node=2 \
    --master_port=7777 DEIMv2/train.py \
    -c DEIMv2/configs/deimv2/deimv2_finetune.yml \
    --tuning deimv2.pth \
    --use-amp --seed=0

W0707 14:26:08.897000 139 torch/distributed/run.py:793] 
W0707 14:26:08.897000 139 torch/distributed/run.py:793] *****************************************
W0707 14:26:08.897000 139 torch/distributed/run.py:793] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0707 14:26:08.897000 139 torch/distributed/run.py:793] *****************************************
2026-07-07 14:26:13.406156: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-07 14:26:13.406158: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783434373.620617     144 cuda_dnn.cc:8579] Unabl

In [11]:
!python DEIMv2/train.py \
  -c DEIMv2/configs/deimv2/deimv2_finetune.yml \
  -t outputs/best_stg2.pth \
  --test-only \
  -u val_dataloader.dataset.ann_file=dataset/annotations/instances_test.json val_dataloader.dataset.img_folder=dataset/images/test \
  > outputs/test_results.txt

2026-07-07 21:24:14.844472: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783459454.868510    1978 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783459454.875658    1978 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783459454.895132    1978 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783459454.895167    1978 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783459454.895170    1978 computation_placer.cc:177] computation placer alr

In [12]:
!mv outputs/best_stg2.pth outputs/deimv2_anime.pth

In [13]:
!sed -i 's/torch.rand(32,/torch.rand(1,/g' DEIMv2/tools/deployment/export_onnx.py

In [14]:
!python DEIMv2/tools/deployment/export_onnx.py --check -c DEIMv2/configs/deimv2/deimv2_finetune.yml -r outputs/deimv2_anime.pth

2026-07-07 21:27:30.145196: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783459650.167339    2017 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783459650.175061    2017 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783459650.192388    2017 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783459650.192606    2017 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783459650.192610    2017 computation_placer.cc:177] computation placer alr

In [15]:
import os
import shutil

FILES_TO_KEEP = [
    "outputs/deimv2_anime.onnx",
    "outputs/deimv2_anime.pth",
    "outputs/test_results.txt",
    "outputs/log.txt"
]

BASE_DIR = os.getcwd()

def clean_dir():
    saved_buffers = {}

    for rel_path in FILES_TO_KEEP:
        full_path = os.path.join(BASE_DIR, rel_path)
        if os.path.exists(full_path) and os.path.isfile(full_path):
            try:
                with open(full_path, 'rb') as f:
                    saved_buffers[os.path.basename(rel_path)] = f.read()
                print(f"Staged for preservation: {rel_path}")
            except Exception as e:
                print(f"Failed to read {rel_path}: {e}")
        else:
            print(f"Warning: Target file not found: {rel_path}")

    for item in os.listdir(BASE_DIR):
        item_path = os.path.join(BASE_DIR, item)
        try:
            if os.path.isfile(item_path) or os.path.islink(item_path):
                os.remove(item_path)
                print(f"Deleted file: {item_path}")
            elif os.path.isdir(item_path):
                shutil.rmtree(item_path)
                print(f"Deleted folder: {item_path}")
        except Exception as e:
            print(f"Failed to delete {item_path}: {e}")

    print("\nRestoring preserved files to root...")
    for filename, data in saved_buffers.items():
        destination_path = os.path.join(BASE_DIR, filename)
        try:
            with open(destination_path, 'wb') as f:
                f.write(data)
            print(f"Restored (flattened): {destination_path}")
        except Exception as e:
            print(f"Failed to restore {filename}: {e}")

if __name__ == "__main__":
    clean_dir()

Staged for preservation: outputs/deimv2_anime.onnx
Staged for preservation: outputs/deimv2_anime.pth
Staged for preservation: outputs/test_results.txt
Staged for preservation: outputs/log.txt
Deleted folder: /kaggle/working/dataset
Deleted folder: /kaggle/working/outputs
Deleted file: /kaggle/working/deimv2_finetune.yml
Deleted file: /kaggle/working/__notebook__.ipynb
Deleted file: /kaggle/working/__huggingface_repos__.json
Deleted folder: /kaggle/working/ckpts
Deleted file: /kaggle/working/deimv2.pth
Deleted folder: /kaggle/working/DEIMv2
Deleted file: /kaggle/working/danbooru-annotated-images.zip

Restoring preserved files to root...
Restored (flattened): /kaggle/working/deimv2_anime.onnx
Restored (flattened): /kaggle/working/deimv2_anime.pth
Restored (flattened): /kaggle/working/test_results.txt
Restored (flattened): /kaggle/working/log.txt
